In [3]:
!git clone -b keypoint-tracker-expt https://github.com/SairajLoke/ori.git

Cloning into 'ori'...
remote: Enumerating objects: 1245, done.
remote: Counting objects: 100% (787/787), done.
remote: Compressing objects: 100% (450/450), done.
remote: Total 1245 (delta 410), reused 695 (delta 329), pack-reused 458 (from 1)
Receiving objects: 100% (1245/1245), 82.86 MiB | 24.86 MiB/s, done.
Resolving deltas: 100% (476/476), done.
Updating files: 100% (1187/1187), done.


In [ ]:
%cd /content/ori
!git pull origin keypoint-tracker-expt
%cd keypoint_expt
!pip install -q -r requirements.txt  # torch already present w/ CUDA on Colab, not reinstalled
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

hf_token = ""
repo_id = "SharpaIT/Robotic_Origami_Challenge"
revision = "competition-paper-set"

api = HfApi()
repo_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", revision=revision, token=hf_token)
seasons = sorted({p.split("/")[0] for p in repo_files if "/" in p})
season_name = seasons[1]
print(f"{len(seasons)} seasons found, downloading: {season_name}")

local_dir = "/content/Robotic_Origami_Challenge"
snapshot_download(repo_id=repo_id, repo_type="dataset", revision=revision,
                  allow_patterns=f"{season_name}/*", local_dir=local_dir, token=hf_token)

SEASON_ROOT = f"{local_dir}/{season_name}"
print(SEASON_ROOT)

39 seasons found, downloading: season_POC22061_2026_07_14_10_09_42_train


Fetching ... files: 0it [00:00, ?it/s]

/content/Robotic_Origami_Challenge/season_POC22061_2026_07_14_10_09_42_train


In [9]:
import json, glob, os

info_path = glob.glob(f"{SEASON_ROOT}/**/meta/info.json", recursive=True)[0]
lerobot_root = os.path.dirname(os.path.dirname(info_path))
info = json.load(open(info_path))

print("root:", lerobot_root)
print("total_episodes:", info.get("total_episodes"))
print("total_frames:", info.get("total_frames"))
print("fps:", info.get("fps"))
for k, v in info.get("features", {}).items():
    print(f"  {k:38s} shape={v.get('shape')} dtype={v.get('dtype')}")

os.environ["DATASET_ROOT"] = lerobot_root
os.environ["MAX_EPISODES"] = str(info.get("total_episodes", 0))
os.environ["VAL_EPISODES"] = "0,1"
print("DATASET_ROOT =", lerobot_root)

root: /content/Robotic_Origami_Challenge/season_POC22061_2026_07_14_10_09_42_train/lerobot3.0
total_episodes: 6
total_frames: 29213
fps: 30
  observation.state                      shape=[65] dtype=float32
  observation.state.joint_torque         shape=[65] dtype=float32
  observation.state.tcp                  shape=[24] dtype=float32
  action                                 shape=[65] dtype=float32
  observation.images.head_left           shape=[480, 480, 3] dtype=video
  observation.images.wrist_left          shape=[480, 480, 3] dtype=video
  observation.images.wrist_right         shape=[480, 480, 3] dtype=video
  observation.images.head_right          shape=[480, 480, 3] dtype=video
  observation.tactile                    shape=[60] dtype=float32
  observation.images.tactile_deform      shape=[480, 1200, 3] dtype=video
  observation.images.tactile_raw         shape=[480, 1600, 3] dtype=video
  timestamp                              shape=[1] dtype=float32
  frame_index            

In [ ]:
%cd ori 

/content/ori


In [13]:
!git pull 

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 3.31 KiB | 1.10 MiB/s, done.
From https://github.com/SairajLoke/ori
   c352e31..69aee96  keypoint-tracker-expt -> origin/keypoint-tracker-expt
Updating c352e31..69aee96
Fast-forward
 .gitignore                            |   1 +
 keypoint_expt/run_cotracker.py        |  15 +--
 vitacformer++/vitacformer_colab.ipynb | 185 ++++++++++++++++++++++++++++++++--
 3 files changed, 188 insertions(+), 13 deletions(-)


In [14]:
import glob, os

# DATASET_ROOT was set by the previous cell after the HF download.
head_left_videos = sorted(glob.glob(f"{os.environ['DATASET_ROOT']}/videos/observation.images.head_left/**/*.mp4", recursive=True))
wrist_left_videos = sorted(glob.glob(f"{os.environ['DATASET_ROOT']}/videos/observation.images.wrist_left/**/*.mp4", recursive=True))
print(f"{len(head_left_videos)} head_left video file(s), {len(wrist_left_videos)} wrist_left video file(s)")
print("using:", head_left_videos[0], "/", wrist_left_videos[0])

2 head_left video file(s), 2 wrist_left video file(s)
using: /content/Robotic_Origami_Challenge/season_POC22061_2026_07_14_10_09_42_train/lerobot3.0/videos/observation.images.head_left/chunk-000/file-000.mp4 / /content/Robotic_Origami_Challenge/season_POC22061_2026_07_14_10_09_42_train/lerobot3.0/videos/observation.images.wrist_left/chunk-000/file-000.mp4


In [16]:
%cd keypoint_expt

/content/ori/keypoint_expt


In [17]:
!python run_cotracker.py --video "{head_left_videos[0]}" --out_name full_head_left --grid_size 20

device: cuda (Tesla T4)
loading CoTracker3 (online) via torch.hub...
Downloading: "https://github.com/facebookresearch/co-tracker/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_online.pth" to /root/.cache/torch/hub/checkpoints/scaled_online.pth
100% 97.0M/97.0M [00:01<00:00, 96.9MB/s]
reading /content/Robotic_Origami_Challenge/season_POC22061_2026_07_14_10_09_42_train/lerobot3.0/videos/observation.images.head_left/chunk-000/file-000.mp4 from t=0.0s (full file) ...


: 

: 

In [ ]:
!python run_cotracker.py --video "{wrist_left_videos[0]}" --out_name full_wrist_left --grid_size 20

In [ ]:
!ls -la results/

In [ ]:
%cd /content
!git clone -q https://github.com/facebookresearch/sam2.git
%cd sam2 && pip install -q -e . && cd checkpoints && ./download_ckpts.sh && cd ../..
%cd /content/ori/keypoint_expt

In [ ]:
# Step 1: dump frame 0 so you can pick a paper pixel by eye
!python run_sam2_paper_mask.py --video "{head_left_videos[0]}" --out_name head_left --dump_first_frame

from IPython.display import Image, display
display(Image(filename="results/head_left_frame0.png"))

In [ ]:
# Step 2: fill in X, Y from the image above (a pixel clearly on the paper), then run.
!python run_sam2_paper_mask.py --video "{head_left_videos[0]}" --out_name head_left --point X Y